# Сравнение методов извлечения числа мест из документов

По текущему базовому пайплайну, если не находится число мест в названии тендера, то нужно идти искать в PDF-документы. Здесь проверяю, насколько хорошо это работает и пробую улучшения:

1. **Baseline** — то, что уже есть
2. **Улучшенный regex** — точечный поиск по паттернам + блэклист ловушек
3. **LLM на сниппете** — та же короткая выдержка, но отдаю Mistral
4. **LLM на полном документе** — для случаев где сниппет нерелевантный

### Загрузка данных

In [35]:
import pandas as pd
import numpy as np
import re
import ssl
import json as _json
import time
import urllib.request
import fitz
from pathlib import Path

ROOT = Path(".")

Золотой стандарт — то, что я вручную проверила и нашла 25 ошибок (при извлечении из doc)

In [36]:
val = pd.read_csv(ROOT / "data_processed/places_validation_filled.csv")
val = val[val["is_correct"].notna()].copy()
val["places_gold"] = pd.to_numeric(val["places_corrected"], errors="coerce")
val["places_auto"] = pd.to_numeric(val["places"], errors="coerce")


Делю на 2 группы:
1. title: число мест было прямо в названии тендера («Школа на N мест»)
2. doc: пришлось идти в документ

In [37]:
in_sample = val["in_working_sample"] == True
doc_mask  = val["source_type"] == "doc"

doc_sub = val[in_sample & doc_mask].copy().reset_index(drop=True)

print(f"Всего проверено: {in_sample.sum()}")
print(f"  из названия (title): {(in_sample & ~doc_mask).sum()}")
print(f"  из документов (doc): {(in_sample & doc_mask).sum()}")

Всего проверено: 78
  из названия (title): 42
  из документов (doc): 36


«Верно» = предсказание отличается от правильного не больше чем на 1%

In [38]:
def accuracy(pred, gold):
    mask = pred.notna() & gold.notna()
    if mask.sum() == 0:
        return 0.0, 0
    correct = (abs(pred[mask] - gold[mask]) / gold[mask].clip(lower=1) <= 0.01)
    return float(correct.mean()), int(mask.sum())

### Baseline — текущий пайплайн

In [39]:
sub = val[in_sample]

a_title, n_title = accuracy(sub[~doc_mask]["places_auto"], sub[~doc_mask]["places_gold"])
a_doc, n_doc = accuracy(sub[ doc_mask]["places_auto"], sub[ doc_mask]["places_gold"])
a_all, n_all = accuracy(sub["places_auto"], sub["places_gold"])

print(f"Из названия (n={n_title}): {a_title:.0%}")
print(f"Из документа (n={n_doc}): {a_doc:.0%}")
print(f"Итого (n={n_all}): {a_all:.0%}")

Из названия (n=42): 100%
Из документа (n=36): 28%
Итого (n=78): 67%


Из названия — 100%: там явно написано «Школа на N мест», regex работает идеально

Из документов — только 31% правильно отработало. Пайплайн берет первое число рядом со словом «мест», поэтому часто попадает не туда: электромощность в кВт, почтовый индекс, норматив из СП

Примеры ошибок: `423 кВт -> 423` вместо `1175 мест`, `21847 (ИКЗ) → 21847` вместо `550`, иногда извлекаются числа из адресов

### Улучшенный regex

Паттерны, которые с высокой вероятностью указывают на число мест школы:

In [40]:
GOOD = [
    r'на\s+(\d[\d\s]{0,4}\d+)\s*мест',
    r'вместимост\w+\s*[–—:\-]\s*(\d[\d\s]{0,4}\d+)',
    r'(\d[\d\s]{0,4}\d+)\s*(?:ученических\s+мест|учащихся|обучающихся)',
    r'(?:количество|число)\s+мест\s*[–—:\-]\s*(\d[\d\s]{0,4}\d+)',
    r'мощност\w+\s+(?:объекта|школы)?\s*[–—:\-]?\s*(\d[\d\s]{0,4}\d+)\s*мест',
]

Если в сниппете встречается одно из этих — лучше вернуть None, чем ошибиться

In [41]:
BAD = [
    r'кВт',
    r'(?:м²|м2|кв\.?\s*м)',
    r'мест(?:ах)?\s+массового',
    r'код(?:а)?\s+закупки',
    r'на одного\s+(?:человека|обучающегося)',
]

In [42]:
def improved_regex(snippet, fallback=None):
    if not isinstance(snippet, str) or not snippet.strip():
        return fallback

    for pat in BAD:
        if re.search(pat, snippet, re.IGNORECASE):
            return None

    for pat in GOOD:
        m = re.search(pat, snippet, re.IGNORECASE)
        if m:
            try:
                return float(re.sub(r'\s+', '', m.group(1)))
            except ValueError:
                continue
    return fallback

doc_sub["places_improved"] = doc_sub.apply(
    lambda r: improved_regex(r["places_snippet"], r["places_auto"]), axis=1)

# Там, где улучшенный regex вернул None — оставляем baseline
doc_sub["places_imp_fill"] = doc_sub["places_improved"].fillna(doc_sub["places_auto"])

a_imp_ans, n_imp_ans = accuracy(doc_sub["places_improved"],  doc_sub["places_gold"])
a_imp_all, _ = accuracy(doc_sub["places_imp_fill"],  doc_sub["places_gold"])

print(f"Improved regex там, где дает ответ (n={n_imp_ans}): {a_imp_ans:.0%}")
print(f"Improved regex + baseline fallback (все 36): {a_imp_all:.0%}")

Improved regex там, где дает ответ (n=24): 62%
Improved regex + baseline fallback (все 36): 44%


Стало лучше: 47% vs 31%, но всё ещё много null — regex не умеет понимать контекст

### LLM на сниппете

Та же выдержка, что и у baseline (snippet), только теперь отдаю Mistral

In [43]:
LLM_API_KEY = open(ROOT / ".llm_key").read().strip()
LLM_MODEL = "mistral-small-latest"
LLM_URL = "https://api.mistral.ai/v1/chat/completions"

def ask_llm(pub_name, snippet):
    prompt = (
        "Ты анализируешь тендерную документацию на строительство школы.\n\n"
        f"Название публикации: {pub_name}\n"
        f"Фрагмент документации: {snippet}\n\n"
        "Задача: определи мощность школы — количество ученических мест.\n"
        "Важно: кВт — это электросеть, м² — площадь, не путай с местами для учеников.\n"
        "Также может быть указано количество учеников на класс или другое помещение / зал, не путай такое количество с общим количеством ученических мест на школу.\n"
        'Ответь строго в формате JSON: {"places": <целое число> или null}\n'
        "null — если в тексте нет однозначного числа мест."
    )
    data = _json.dumps({
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
    }).encode()
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(
        LLM_URL, data=data,
        headers={"Authorization": f"Bearer {LLM_API_KEY}", "Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
        raw = _json.loads(resp.read())["choices"][0]["message"]["content"]
    parsed = _json.loads(raw.strip().strip("```json").strip("```").strip())
    v = parsed.get("places")
    return float(v) if v is not None else None

In [44]:
llm_snippet_results = []

for i, row in doc_sub.iterrows():
    llm_val = ask_llm(row["pub_name"], str(row["places_snippet"]))
    time.sleep(0.5)

    gold = row["places_gold"]
    ok = "✓" if (llm_val is not None and abs(llm_val - gold) / max(gold, 1) < 0.01) else (
         "∅" if llm_val is None else "✗")
    print(f"{i+1:2d}/36 {ok}  LLM={llm_val}  gold={gold:.0f}")

    llm_snippet_results.append({
        "registry_number": row["registry_number"],
        "pub_name": row["pub_name"],
        "places_snippet": row["places_snippet"],
        "places_llm": llm_val,
        "places_corrected":row["places_gold"],
        "places_imp_fill": row["places_imp_fill"],
    })

llm_df = pd.DataFrame(llm_snippet_results)
llm_df.to_csv(ROOT / "data_processed/places_llm_results.csv", index=False)

 1/36 ∅  LLM=None  gold=202
 2/36 ∅  LLM=None  gold=1175
 3/36 ∅  LLM=None  gold=1175
 4/36 ∅  LLM=None  gold=1175
 5/36 ∅  LLM=None  gold=1175
 6/36 ∅  LLM=None  gold=1500
 7/36 ∅  LLM=None  gold=1100
 8/36 ✓  LLM=1280.0  gold=1280
 9/36 ✓  LLM=500.0  gold=500
10/36 ∅  LLM=None  gold=500
11/36 ✗  LLM=150.0  gold=100
12/36 ∅  LLM=None  gold=550
13/36 ✓  LLM=1224.0  gold=1224
14/36 ✓  LLM=1224.0  gold=1224
15/36 ✓  LLM=1224.0  gold=1224
16/36 ✓  LLM=1224.0  gold=1224
17/36 ✓  LLM=1050.0  gold=1050
18/36 ∅  LLM=None  gold=300
19/36 ✓  LLM=100.0  gold=100
20/36 ✓  LLM=1050.0  gold=1050
21/36 ∅  LLM=None  gold=300
22/36 ∅  LLM=None  gold=825
23/36 ✓  LLM=1050.0  gold=1050
24/36 ∅  LLM=None  gold=825
25/36 ∅  LLM=None  gold=1050
26/36 ∅  LLM=None  gold=1440
27/36 ∅  LLM=None  gold=60
28/36 ✗  LLM=625.0  gold=550
29/36 ✓  LLM=180.0  gold=180
30/36 ∅  LLM=None  gold=1100
31/36 ∅  LLM=None  gold=500
32/36 ∅  LLM=None  gold=520
33/36 ✓  LLM=825.0  gold=825
34/36 ✓  LLM=60.0  gold=60
35/36 ∅  LL

In [45]:
llm_df["places_llm"] = pd.to_numeric(llm_df["places_llm"], errors="coerce")

a_llm, n_llm = accuracy(llm_df["places_llm"], llm_df["places_corrected"])
n_null = llm_df["places_llm"].isna().sum()

print(f"LLM ответил на {n_llm} из 36 случаев (в {n_null} вернул null)")
print(f"Точность там, где ответил: {a_llm:.0%}")

LLM ответил на 15 из 36 случаев (в 21 вернул null)
Точность там, где ответил: 87%


Где LLM вернул null (не смог определить по сниппету):

In [46]:
for _, r in llm_df[llm_df["places_llm"].isna()].iterrows():
    print(f"gold={r['places_corrected']:.0f}  | {str(r['pub_name'])[:60]}  | {r['places_snippet']}")

gold=202  | Выполнение строительно-монтажных, пусконаладочных работ, пос  | основании _____________, с другой стороны, вместе именуемые «Стороны», в соответствии с протоколом____________________________ от «___»___________202_ г. № ___________, ИКЗ
gold=1175  | Выполнение строительно-монтажных, пусконаладочных работ, пос  | 3. Макс. мощность присоединяемых энергопринимающих устройств заявителя составляет 423,6 кВт____ (если энергопринимающее
gold=1175  | Выполнение строительно-монтажных, пусконаладочных работ, пос  | 3. Макс. мощность присоединяемых энергопринимающих устройств заявителя составляет 423,6 кВт____ (если энергопринимающее
gold=1175  | Выполнение строительно-монтажных, пусконаладочных работ, пос  | 3. Макс. мощность присоединяемых энергопринимающих устройств заявителя составляет 423,6 кВт____ (если энергопринимающее
gold=1175  | Выполнение строительно-монтажных, пусконаладочных работ, пос  | 3. Макс. мощность присоединяемых энергопринимающих устройств заявителя составляет 4

LLM отвечает точнее regex в 2.5 раза (~81%), но в ~20 случаях она вернула `null` — сниппет нерелевантный (кВт, почтовый индекс, шаблон договора), модель правильно решила не угадывать

### LLM на полном тексте документа

Для null проблема не в LLM, а в извлеченном сниппете — пайплайн выбрал не тот кусок, поэтому теперь перебираю все PDF в папке тендера, ищу раздел с числом мест (ТЭП, ТЗ, заключение ГЭ) и отдаю этот контекст в LLM

In [ ]:
DOCS_DIR = ROOT / "docs"

SEARCH = re.compile(
    r'(?:вместимост\w+|мест(?:а|ах)?\s+(?:обучающихся|учащихся|школы)'
    r'|учащихся[\s,]*\d|на\s+\d+\s*мест|\d+\s*(?:учащихся|мест(?:а)?)[\s,])',
    re.IGNORECASE
)

import docx as _docx

def _text_from_pdf(path):
    doc = fitz.open(str(path))
    return "\n".join(page.get_text() for page in doc)

def _text_from_docx(path):
    try:
        doc = _docx.Document(str(path))
        parts = [p.text for p in doc.paragraphs]
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    parts.append(cell.text)
        return "\n".join(parts)
    except Exception:
        return ""

def find_snippet_in_docs(registry_number):
    """Ищет релевантный фрагмент в PDF и DOCX файлах папки тендера"""
    rn = str(registry_number)
    folders = [d for d in DOCS_DIR.iterdir() if d.name.startswith(rn)]
    if not folders:
        return None, None

    ext_dir = folders[0] / "extracted"

    candidates = sorted(ext_dir.rglob("*.pdf")) + \
                 sorted(ext_dir.rglob("*.docx")) + \
                 sorted(ext_dir.rglob("*.doc"))

    for fpath in candidates:
        if fpath.suffix.lower() == ".pdf":
            text = _text_from_pdf(fpath)
        else:
            text = _text_from_docx(fpath)

        m = SEARCH.search(text)
        if m:
            snippet = text[max(0, m.start()-50) : m.start()+300].strip()
            return fpath.name, snippet

    return None, None

Прогон LLM только на null (где сниппет нерелевантный):

In [48]:
null_rns = llm_df[llm_df["places_llm"].isna()]["registry_number"].unique()
print(f"Null: {len(null_rns)} уникальных тендеров")

Null: 21 уникальных тендеров


In [49]:
ft_results = []

for i, rn in enumerate(null_rns):
    fname, snip = find_snippet_in_docs(rn)

    row = llm_df[llm_df["registry_number"] == rn].iloc[0]
    pub_name = row["pub_name"]
    gold = row["places_corrected"]

    if snip:
        llm_val = ask_llm(pub_name, snip)
        time.sleep(0.5)
    else:
        llm_val = None

    ok = "✓" if (llm_val is not None and abs(llm_val - gold) / max(gold, 1) < 0.01) else (
         "∅" if llm_val is None else "✗")
    print(f"{i+1:2d}/{len(null_rns)} {ok}  LLM_ft={llm_val}  gold={gold:.0f}  файл={fname}")

    ft_results.append({
        "registry_number": rn,
        "ft_file": fname,
        "places_llm_ft": llm_val,
        "places_gold": gold,
    })

ft_df = pd.DataFrame(ft_results)
ft_df.to_csv(ROOT / "data_processed/places_llm_fulltext_results.csv", index=False)

 1/21 ∅  LLM_ft=None  gold=202  файл=None
 2/21 ✓  LLM_ft=1175.0  gold=1175  файл=ПЗ ГЭ (тех. часть) от 24.12.2019.pdf
 3/21 ✓  LLM_ft=1175.0  gold=1175  файл=ПЗ ГЭ (тех. часть) от 24.12.2019.pdf
 4/21 ✓  LLM_ft=1175.0  gold=1175  файл=ПЗ ГЭ (тех. часть) от 24.12.2019.pdf
 5/21 ✓  LLM_ft=1175.0  gold=1175  файл=ПЗ ГЭ (тех. часть) от 24.12.2019.pdf
 6/21 ∅  LLM_ft=None  gold=1500  файл=None
 7/21 ∅  LLM_ft=None  gold=1100  файл=None
 8/21 ✓  LLM_ft=500.0  gold=500  файл=!Å«ßΓá¡«ó½Ñ¡¿Ñ áñ¼¿¡¿ßΓαáµ¿¿ ú«α«ñá ì¿ª¡Ñú« ì«óú«α«ñá_(Σá⌐½ «Γ«íαáªÑ¡¿∩) (1).pdf
 9/21 ∅  LLM_ft=None  gold=550  файл=Задание на проектирование.docx
10/21 ✓  LLM_ft=300.0  gold=300  файл=Приложение № 1 к ТЗ Отраслевое техническое задание.pdf
11/21 ✓  LLM_ft=300.0  gold=300  файл=Приложение № 1 к ТЗ Отраслевое техническое задание.pdf
12/21 ✓  LLM_ft=825.0  gold=825  файл=Заключение экспертизы.pdf
13/21 ✓  LLM_ft=825.0  gold=825  файл=Заключение экспертизы.pdf
14/21 ✓  LLM_ft=1050.0  gold=1050  файл=Приложение № 1 к ГК - Т

In [50]:
ft_df["places_llm_ft"] = pd.to_numeric(ft_df["places_llm_ft"], errors="coerce")

a_ft, n_ft = accuracy(ft_df["places_llm_ft"], ft_df["places_gold"])
n_ft_null = ft_df["places_llm_ft"].isna().sum()

print(f"LLM на полном тексте: ответил на {n_ft} из {len(ft_df)}, верно {a_ft:.0%}")
print(f"Не смог ответить: {n_ft_null}")

LLM на полном тексте: ответил на 13 из 21, верно 92%
Не смог ответить: 8


### Итоговое сравнение всех методов

LLM_snip -> LLM_fulltext -> improved_regex -> baseline

In [51]:
ft_map = dict(zip(ft_df["registry_number"].astype(str), ft_df["places_llm_ft"]))
llm_map = dict(zip(llm_df["registry_number"].astype(str), llm_df["places_llm"]))

doc_sub["places_llm_snip"] = doc_sub["registry_number"].astype(str).map(llm_map)
doc_sub["places_llm_snip"] = pd.to_numeric(doc_sub["places_llm_snip"], errors="coerce")

doc_sub["places_llm_ft"] = doc_sub["registry_number"].astype(str).map(ft_map)
doc_sub["places_llm_ft"] = pd.to_numeric(doc_sub["places_llm_ft"], errors="coerce")

def best_value(row):
    if pd.notna(row["places_llm_snip"]): return row["places_llm_snip"]
    if pd.notna(row["places_llm_ft"]): return row["places_llm_ft"]
    if pd.notna(row["places_imp_fill"]): return row["places_imp_fill"]
    return row["places_auto"]

doc_sub["places_final"] = doc_sub.apply(best_value, axis=1)

a_base, _ = accuracy(doc_sub["places_auto"], doc_sub["places_gold"])
a_imp, _ = accuracy(doc_sub["places_imp_fill"], doc_sub["places_gold"])
a_snip, _ = accuracy(doc_sub["places_llm_snip"], doc_sub["places_gold"])
a_ft2, _ = accuracy(doc_sub[doc_sub["places_llm_snip"].isna()]["places_llm_ft"],
                     doc_sub[doc_sub["places_llm_snip"].isna()]["places_gold"])
a_fin, _ = accuracy(doc_sub["places_final"], doc_sub["places_gold"])

n_t, n_d, n_tot = 42, 36, 78 # title/doc/всего в рабочей выборке

print(f"{'Метод':<45} {'doc':>5}  {'overall':>8}")
print("-" * 61)
print(f"{'Baseline (текущий regex)':<45} {a_base:>5.0%}  {(n_t + n_d*a_base)/n_tot:>8.0%}")
print(f"{'Improved regex + fallback':<45} {a_imp:>5.0%}  {(n_t + n_d*a_imp)/n_tot:>8.0%}")
print(f"{'LLM на snippet (n=16 ответивших)':<45} {a_snip:>5.0%}  {'—':>8}")
print(f"{'LLM на полном тексте (' + str(n_ft) + ' ответивших)':<45} {a_ft2:>5.0%}  {'—':>8}")
print(f"{'LLM_snip -> LLM_ft -> regex -> baseline':<45} {a_fin:>5.0%}  {(n_t + n_d*a_fin)/n_tot:>8.0%}")

Метод                                           doc   overall
-------------------------------------------------------------
Baseline (текущий regex)                        28%       67%
Improved regex + fallback                       44%       74%
LLM на snippet (n=16 ответивших)                87%         —
LLM на полном тексте (13 ответивших)            92%         —
LLM_snip -> LLM_ft -> regex -> baseline         75%       88%


### Выводы

* LLM на snippet — высокое качетво - 87% на doc-случаях. Модель понимает контекст и не путает вместимость школы с другими упоминаниями «мест»

* LLM на полном тексте документа показала 92% на 13 ответивших случаях — выше, чем на snippet - это объясняется тем, что после добавления поддержки .docx (включая таблицы) функция стала находить файлы с четкими техническими данными (ТЗ, обоснование НМЦК), где вместимость указана явно

* Regex работает плохо (28%): берет первое число рядом со словом «мест» без понимания контекста — попадает на залы, парковки, проценты. Улучшенный regex с фильтрами поднимает до 44%, но все равно в 2.5 раза хуже LLM

Итог: каскад всех методов дает 88% overall против 67% у baseline (+21 пп)

**Оптимальный пайплайн:**
1. Если число есть в названии публикации — берем оттуда (100%)
2. Для doc-случаев: LLM на snippet -> 87% из ответивших
3. Для null после шага 2: искать в PDF + DOCX (параграфы и таблицы) → LLM -> 92%
4. Fallback: улучшенный regex (44%)